Text Cleaning

In [3]:
import pandas as pd
df = pd.read_csv('resume_data.csv')
print(df.columns)

Index(['address', 'career_objective', 'skills', 'educational_institution_name',
       'degree_names', 'passing_years', 'educational_results', 'result_types',
       'major_field_of_studies', 'professional_company_names', 'company_urls',
       'start_dates', 'end_dates', 'related_skils_in_job', 'positions',
       'locations', 'responsibilities', 'extra_curricular_activity_types',
       'extra_curricular_organization_names',
       'extra_curricular_organization_links', 'role_positions', 'languages',
       'proficiency_levels', 'certification_providers', 'certification_skills',
       'online_links', 'issue_dates', 'expiry_dates', '﻿job_position_name',
       'educationaL_requirements', 'experiencere_requirement',
       'age_requirement', 'responsibilities.1', 'skills_required',
       'matched_score'],
      dtype='object')


In [7]:
import re

columns_to_combine = ['career_objective', 'skills', 'major_field_of_studies', 'responsibilities']
df['combined_text'] = df[columns_to_combine].fillna('').agg(' '.join, axis=1)

def clean_data(text):
    text = re.sub(r'http\S+\s*', ' ', text)  
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df['cleaned_resume'] = df['combined_text'].apply(clean_data)

<>:8: SyntaxWarning: invalid escape sequence '\]'
<>:8: SyntaxWarning: invalid escape sequence '\]'
C:\Users\user\AppData\Local\Temp\ipykernel_22920\4176133366.py:8: SyntaxWarning: invalid escape sequence '\]'
  text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)


In [9]:
!pip install sentence-transformers

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

resume_embeddings = model.encode(df['cleaned_resume'].tolist(), show_progress_bar=True)

print(f"Created embeddings with shape: {resume_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

e:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Created embeddings with shape: (9544, 384)


Ranking & Justification

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

job_description = "We are looking for a Data Scientist skilled in Python, NLP, and Machine Learning."

cleaned_jd = clean_data(job_description) 
jd_embedding = model.encode([cleaned_jd])
 
scores = cosine_similarity(jd_embedding, resume_embeddings)[0]
df['match_score'] = scores

Present Top-Ranked Results with Justification

In [12]:
top_3 = df.sort_values(by='match_score', ascending=False).head(3)

print("--- Top Ranked Resumes ---")
for i, row in top_3.iterrows():
    print(f"\nMatch Score: {round(row['match_score'] * 100, 2)}%")
    print(f"Skills: {row['skills']}")
    
    print(f"Justification: This candidate's profile has a {round(row['match_score']*100)}% semantic overlap "
          f"with the job requirements, specifically matching key skills in their profile.")

--- Top Ranked Resumes ---

Match Score: 67.2%
Skills: ['Machine Learning', 'Data Transition', 'Natural Language Processing', 'Predictive Analytics', 'Data Cleaning', 'Data Visualization', 'Deep Learning', 'BeRT Modelling', 'Tensorflow', 'Python', 'SciKit Learn', 'Data computation libraries in python']
Justification: This candidate's profile has a 67% semantic overlap with the job requirements, specifically matching key skills in their profile.

Match Score: 67.06%
Skills: ['Machine Learning', 'Data Transition', 'Natural Language Processing', 'Predictive Analytics', 'Data Cleaning', 'Data Visualization', 'Deep Learning', 'BeRT Modelling', 'Tensorflow', 'Python', 'SciKit Learn', 'Data computation libraries in python']
Justification: This candidate's profile has a 67% semantic overlap with the job requirements, specifically matching key skills in their profile.

Match Score: 67.01%
Skills: ['Machine Learning', 'Data Transition', 'Natural Language Processing', 'Predictive Analytics', 'Dat

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

job_description = "Looking for a software engineer with skills in Python, SQL, and cloud platforms."
cleaned_jd = clean_data(job_description)

jd_embedding = model.encode([cleaned_jd])

scores = cosine_similarity(jd_embedding, resume_embeddings)[0]
df['match_score'] = scores

top_ranked = df.sort_values(by='match_score', ascending=False).head(5)

In [14]:
print("--- TASK 8: RANKED CANDIDATES ---")
for i, row in top_ranked.iterrows():
    print(f"\nRANK {i+1}")
    print(f"Match Score: {round(row['match_score'] * 100, 2)}%")
    print(f"Candidate Skills: {row['skills']}")
    
    print(f"Justification: This candidate was ranked highly due to a semantic similarity score of "
          f"{round(row['match_score'], 2)}, indicating their background in {row['major_field_of_studies']} "
          f"aligns well with the job requirements.")

--- TASK 8: RANKED CANDIDATES ---

RANK 5115
Match Score: 69.03%
Candidate Skills: ['Python', 'Visual Studio Code', 'Mercurial']
Justification: This candidate was ranked highly due to a semantic similarity score of 0.69, indicating their background in ['Computer Science'] aligns well with the job requirements.

RANK 3781
Match Score: 65.76%
Candidate Skills: ['Snowflake', 'Python', 'C++', 'R', 'SQL', 'Tableau', 'PyTorch']
Justification: This candidate was ranked highly due to a semantic similarity score of 0.66, indicating their background in ['Engineering'] aligns well with the job requirements.

RANK 5995
Match Score: 65.12%
Candidate Skills: ['Software Development', 'Application Programming', 'System Analysis', 'Technical Architecture', 'Requirement Gathering', 'Client Management', 'Server Management', 'Data Science', 'Machine Learning', 'Python', 'AWS', 'MySQL', 'NoSQL', 'NLP']
Justification: This candidate was ranked highly due to a semantic similarity score of 0.65, indicating th

Streamlit front-end

In [15]:
!pip install streamlit

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.5.1 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 9.10.0 which is incompatible.
spyder-kernels 2.5.0 requires ipykernel<7,>=6.23.2; python_version >= "3.8", but you have ipykernel 7.2.0 which is incompatible.
spyder-kernels 2.5.0 requires ipython!=8.17.1,<9,>=8.13.0; python_version > "3.8", but you have ipython 9.10.0 which is incompatible.


  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

@st.cache_resource 
def load_resources():
    model = SentenceTransformer('all-MiniLM-L6-v2')
    df = pd.read_csv('resume_data.csv')
    return model, df

model, df = load_resources()

def clean_data(text):
    text = re.sub(r'http\S+\s*', ' ', text)  
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

st.title("🚀 AI Resume Screener")
st.subheader("Match Resumes to Job Descriptions Instantly")

with st.sidebar:
    st.header("Job Settings")
    jd_input = st.text_area("Enter Job Description here:", height=300)
    num_results = st.slider("Number of candidates to show", 1, 10, 5)

if st.button("Start Screening"):
    if jd_input:
        with st.spinner("Processing..."):
        
            cleaned_jd = clean_data(jd_input)
            jd_embedding = model.encode([cleaned_jd])

            columns_to_combine = ['career_objective', 'skills', 'major_field_of_studies', 'responsibilities']
            df['combined_text'] = df[columns_to_combine].fillna('').agg(' '.join, axis=1)
            df['cleaned_resume'] = df['combined_text'].apply(clean_data)
            
            resume_embeddings = model.encode(df['cleaned_resume'].tolist())

            scores = cosine_similarity(jd_embedding, resume_embeddings)[0]
            df['match_score'] = scores

            top_ranked = df.sort_values(by='match_score', ascending=False).head(num_results)

            st.success(f"Top {num_results} Matches Found!")
            for i, row in top_ranked.iterrows():
                with st.expander(f"Rank {i+1}: Match Score {round(row['match_score']*100, 2)}%"):
                    st.write(f"**Skills:** {row['skills']}")
                    st.write(f"**Education:** {row['major_field_of_studies']}")
                    st.write(f"**Justification:** High semantic overlap with the job requirements.")
    else:
        st.warning("Please enter a Job Description first!")

Writing app.py
